In [ ]:
#installs all the Python libraries
!pip -q install youtube-transcript-api openpyxl transformers accelerate sentencepiece scikit-learn streamlit yt-dlp imageio-ffmpeg


| Library | Why are we using it in our project? |
|---|---|
| `youtube-transcript-api` | Retrieves available captions or transcripts from YouTube videos. |
| `openpyxl` | Helps Python read and write Excel files (`.xlsx`). |
| `transformers` | Provides pretrained NLP models such as IndicBERT through Hugging Face. |
| `accelerate` | Helps manage and run model training across available hardware, such as GPUs. |
| `sentencepiece` | Provides tokenization support required by some transformer models, including IndicBERT tokenizers. |
| `scikit-learn` | Provides tools for splitting datasets, calculating accuracy, precision, recall, F1-score and other evaluation metrics. |
| `streamlit` | Helps create a web application to display your extracted moments. |

`pipeline.py` is a Python file that contains the functions needed to process YouTube podcast transcripts and identify important conversations.

In [ ]:
%%writefile pipeline.py
"""pipeline.py - multilingual key-moment extraction (shared by the notebook and the Streamlit app).

Moments: question (+answer), agreement, disagreement.
Classifier: IndicBERT fine-tuned on (previous sentence, current sentence) pairs.
Fallback when no fine-tuned model is available: multilingual cue-phrase rules.
"""
import re #	To search for patterns in text and extract YouTube video IDs.
import os, shutil, subprocess, zipfile   # NEW: needed for cutting real video clips
import numpy as np
import pandas as pd

LABELS = ['neutral', 'question', 'agreement', 'disagreement']

#this dictionary connects language names with YouTube's language codes.
LANG_CODES = {
    'english': ['en', 'en-US', 'en-GB', 'en-IN'],
    'hindi': ['hi', 'hi-IN'],
    'malayalam': ['ml', 'ml-IN'],
    'tamil': ['ta', 'ta-IN'],
    'urdu': ['ur', 'ur-PK'],
    'telugu': ['te', 'te-IN'],
    'kannada': ['kn', 'kn-IN'],
    'bengali': ['bn', 'bn-BD', 'bn-IN'],
    'marathi': ['mr', 'mr-IN'],
    'gujarati': ['gu', 'gu-IN'],
    'punjabi': ['pa', 'pa-IN'],
}
#configuration
CFG = dict(
    buffer=25,#before and after detect moment
    min_len=30,#min clip duration 30 sec
    max_len=60,#max clip duration 60 sec
    answer_window=20,#The first answer must start within 20 seconds after a question.
    q_thr=0.60, #	Minimum confidence score for accepting a question prediction.
    ad_thr=0.70, #	Minimum confidence score for accepting agreement/disagreement predictions.
    top_k=8,  #	Maximum 8 selected clips per category per video.
    max_tokens=128)#	Maximum number of tokens given to the model for each sentence pair.

# '.', '?', '!', Devanagari danda, Urdu full stop, Arabic question mark
END_PUNCT = '.?!\u0964\u06D4\u061F'
STRIP_CHARS = '.,!?;:"\'()[]\u0964\u06D4\u061F\u060C-\u2013\u2014\u2026'


def lang_key(name):#checks the language name.
    k = str(name).strip().lower()
    return k if k in LANG_CODES else 'english'
    # Converts it into a string.
    # Removes unnecessary spaces.
    # Converts it to lowercase.
    # Checks whether language exists in LANG_CODES.
    # if language is not in the dict it return english


def extract_video_id(url):#extracts the YouTube video ID from a video link
    m = re.search(r'(?:youtu\.be/|v=|/embed/|/shorts/)([A-Za-z0-9_-]{11})', str(url))
    return m.group(1) if m else None


# Transcript (captions only - no video/audio download)
#retrieves available YouTube captions
def fetch_transcript(yt_id, language='english'):
    """Returns (snippets, language_code_used) or (None, None). Works with youtube-transcript-api 0.x and 1.x."""
    from youtube_transcript_api import YouTubeTranscriptApi
    langs = LANG_CODES[lang_key(language)]#selected vedio language
    try:
        api = YouTubeTranscriptApi()#object
        if hasattr(api, 'fetch'):                                   # v1.x
            try:
                fetched = api.fetch(yt_id, languages=langs)
                #Requests captions for the video in the selected language.
            except Exception:                                       # declared language not found -> any track
                fetched = next(iter(api.list(yt_id))).fetch()
                #It tries to retrieve another available transcript track.
            snips = [{'text': s.text, 'start': s.start, 'duration': s.duration} for s in fetched]
            #       caption,start,end
            return snips, getattr(fetched, 'language_code', None)
        return YouTubeTranscriptApi.get_transcript(yt_id, languages=langs), langs[0]   # v0.x
    except Exception as e:
        print(f'  transcript failed for {yt_id}: {type(e).__name__}: {str(e)[:120]}')
        return None, None


# Caption fragments -> sentences
def to_words(snips):
    words = []
    #Creates an empty list to store individual words.
    for i, s in enumerate(snips):
      #Goes through every caption fragment.
        txt = re.sub(r'\[[^\]]*\]|\([^)]*\)', ' ', s['text']).replace('\n', ' ')
        #remove  [Music], (laughs)
        ws = txt.split()
        #Splits the caption into individual words.
        if not ws:
            continue
        start, dur = s['start'], s['duration']
        #handle overlaping caption
        if i + 1 < len(snips):
            dur = min(dur, max(snips[i + 1]['start'] - start, 0.2))
        dur = max(dur, 0.2)
        for k, w in enumerate(ws):
            words.append({'w': w,#word
                          't': start + dur * k / len(ws), #estimted start time
                          'e': start + dur * (k + 1) / len(ws)}) #estimted end time
    return words

def build_sentences(snips, max_words=30, pause=1.2):
  #max word in group 30,pause of more than 1.2 seconds can indicate a new group
  #combines individual words into sentence-like groups
    words = to_words(snips)
    if not words:
        return [], False
    has_punct = sum(w['w'][-1] in END_PUNCT for w in words) / len(words) > 0.01 ##Why?-it can help identify sentence boundaries.
    sents, cur, turn = [], [], False
#for combiing words use :
  # Punctuation.
  # Time gaps between words.
  # Maximum word limits.
  # Speaker-change markers, if available.


    #sents: Stores the completed sentence groups.
    #cur:   Stores the words in the current group.
    #turn:  Tracks whether a speaker-change marker was detected

    def flush():
        #Saves the collected words as a group
        nonlocal cur, turn
        if cur:
            sents.append({'text': ' '.join(x['w'] for x in cur),
                          'start': cur[0]['t'],
                          'end': cur[-1]['e'],
                          'n': len(cur),
                          'turn': turn})
        cur, turn = [], False

    for w in words:
        tok = w['w']
        if tok.startswith('>>'):  # speaker-change marker
            flush()
            turn = True
            tok = tok.lstrip('>').strip()
            if not tok:
                continue
            w = {**w, 'w': tok}
        elif cur:
            gap = w['t'] - cur[-1]['e'] #previous word - current word
            if (cur[-1]['w'][-1] in END_PUNCT or (not has_punct and (gap > pause or len(cur) >= max_words))
                    or len(cur) >= 45):
                flush()
        cur.append(w)
    flush()
    return sents, has_punct


# Rule-based "weak labels"
#English question words.
EN_WH = r'(what|why|how|when|where|who|whom|whose|which)'
EN_AUX = (r"(do|does|did|is|are|was|were|can|could|would|will|should|have|has|had|am|shall|may|might|"
          r"isn't|aren't|don't|doesn't|didn't|won't)")
EN_Q_START = re.compile(rf'^(so |and |but |okay |ok |well )?({EN_WH}|{EN_AUX})\b', re.I)

#Agreement and disagreement phrases
EN_AGREE = re.compile('|'.join([
    r"i agree", r"i totally agree", r"couldn'?t agree more", r"exactly", r"absolutely", r"you'?re (absolutely )?right",
    r"you are right", r"that'?s (so )?(true|right|correct)", r"definitely", r"for sure", r"well said",
    r"fair (point|enough)", r"i think so too", r"same (here|for me)", r"100 ?%", r"makes sense", r"^yes\b",
    r"^yeah\b", r"^true\b", r"i feel the same"]), re.I)
EN_DISAGREE = re.compile('|'.join([
    r"i disagree", r"i don'?t agree", r"i don'?t think so", r"not really", r"that'?s not (true|right|correct|it)",
    r"i wouldn'?t say", r"i'?m not sure about that", r"on the contrary", r"i beg to differ", r"not necessarily",
    r"i don'?t buy", r"that'?s wrong", r"^no\b", r"^nope\b", r"^but\b", r"^actually\b", r"^however\b",
    r"hold on", r"i('d| would) push back", r"i don'?t think"]), re.I)

#Multilingual cue dictionary
LEX = {
    'hindi': {
        'agree': ['बिल्कुल', 'बिलकुल', 'सही कहा', 'ठीक कहा', 'मैं सहमत हूँ', 'मैं सहमत हूं', 'सही है', 'सच है',
                  'हाँ', 'हां', 'जी हाँ', 'जी हां', 'बिल्कुल सही'],
        'disagree': ['सहमत नहीं', 'असहमत', 'नहीं', 'ऐसा नहीं है', 'ऐसा नहीं', 'मुझे नहीं लगता', 'लेकिन', 'पर',
                     'गलत', 'ग़लत'],
        'question': ['क्या', 'कैसे', 'क्यों', 'कब', 'कहाँ', 'कहां', 'कौन', 'कितना', 'कितने', 'किस', 'किसने']},
    'urdu': {
        'agree': ['بالکل', 'جی ہاں', 'جی', 'ٹھیک ہے', 'درست', 'میں متفق ہوں', 'آپ صحیح کہہ رہے ہیں', 'صحیح کہا'],
        'disagree': ['متفق نہیں', 'اختلاف', 'نہیں', 'ایسا نہیں ہے', 'ایسا نہیں', 'مجھے نہیں لگتا', 'لیکن', 'غلط'],
        'question': ['کیا', 'کیسے', 'کیوں', 'کب', 'کہاں', 'کون', 'کتنا', 'کتنے', 'کس']},
    'tamil': {
        'agree': ['ஆமாம்', 'ஆமா', 'கண்டிப்பா', 'கண்டிப்பாக', 'நிச்சயமா', 'நிச்சயமாக', 'உண்மைதான்', 'சரிதான்',
                  'சரியா சொன்னீங்க', 'நான் ஒத்துக்கிறேன்'],
        'disagree': ['இல்லை', 'இல்ல', 'ஆனா', 'ஆனால்', 'தப்பு', 'தவறு', 'நான் ஒத்துக்கல', 'எனக்கு அப்படி தோணல'],
        'question': ['என்ன', 'எப்படி', 'ஏன்', 'எப்போ', 'எப்போது', 'எங்கே', 'எங்க', 'யார்', 'எவ்வளவு']},
    'malayalam': {
        'agree': ['അതെ', 'ശരിയാണ്', 'ശരിയാ', 'സത്യമാണ്', 'തീർച്ചയായും', 'ഞാൻ യോജിക്കുന്നു'],
        'disagree': ['അല്ല', 'ഇല്ല', 'പക്ഷേ', 'പക്ഷെ', 'തെറ്റാണ്', 'ഞാൻ യോജിക്കുന്നില്ല', 'എനിക്ക് തോന്നുന്നില്ല'],
        'question': ['എന്ത്', 'എന്താണ്', 'എങ്ങനെ', 'എന്തുകൊണ്ട്', 'എപ്പോൾ', 'എവിടെ', 'ആര്', 'എത്ര', 'ആണോ']},
}


def _clean_tokens(text, n=None):
    toks = [t.strip(STRIP_CHARS) for t in text.split()]
    toks = [t for t in toks if t]
    return toks[:n] if n else toks
#It cleans text by removing unnecessary punctuation from the beginning and end of each word.

def _first_hit(text, phrases, head=14):#search only within the first 14 words.
    hay_t = _clean_tokens(text, head)
    #call previous fun [_clean_tokens()]
    #t cleans the sentence and takes the first 14 words.
    hay = ' ' + ' '.join(hay_t) + ' '
    #Joins the cleaned words into one sentence, adding a space at the beginning and end.
    best = None
    for p in phrases:
        idx = hay.find(' ' + p + ' ')
        if idx >= 0:
            pos = hay[:idx + 1].count(' ')
            best = pos if best is None else min(best, pos)
    return best


def weak_label(cur, lang, has_punct, n_words=None):
  #It checks the language, punctuation, question patterns and cue phrases.
  #If it cannot identify any of the special categories, it returns neutral
    """Rule-based label for a sentence: neutral / question / agreement / disagreement."""
    key = lang_key(lang)
    n = n_words if n_words is not None else len(cur.split())
    if '?' in cur or '\u061F' in cur:
        return 'question'
    if key == 'english':
        if not has_punct and EN_Q_START.match(cur.strip()) and 3 <= n <= 28:
            return 'question'
        head = ' '.join(cur.split()[:14])
        a, d = EN_AGREE.search(head), EN_DISAGREE.search(head)
        if a and (not d or a.start() < d.start()):
            return 'agreement'
        if d:
            return 'disagreement'
        return 'neutral'
    lex = LEX.get(key)
    if lex is None:
        return 'neutral'
    toks = set(_clean_tokens(cur))
    if not has_punct and 3 <= n <= 25 and any(q in toks for q in lex['question']):
        return 'question'
    a, d = _first_hit(cur, lex['agree']), _first_hit(cur, lex['disagree'])
    if d is not None and (a is None or d <= a):        # tie -> disagreement ("सहमत नहीं" contains the agree stem)
        return 'disagreement'
    if a is not None:
        return 'agreement'
    return 'neutral'


# Fine-tuned IndicBERT classifier
class MomentClassifier:
  #creates a class responsible for loading the fine-tuned model and using it to classify conversations.
    """Loads a fine-tuned sequence-pair classifier (local folder or Hugging Face repo id)."""

    def __init__(self, path):
        import torch
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        self.torch = torch
        self.tok = AutoTokenizer.from_pretrained(path)
        #Loading the tokenizer
        #text-->numbers
        self.model = AutoModelForSequenceClassification.from_pretrained(path).eval()
        #loads the already fine-tuned classification model from the given path.
        #path can point to a locally saved model folder or a Hugging Face model repository.
        #.eval() puts the model into evaluation mode for prediction.
        self.dev = 'cuda' if torch.cuda.is_available() else 'cpu'#gpu available
        self.model.to(self.dev)#move to selected device
        id2 = self.model.config.id2label
        self.labels = [id2[i] for i in range(len(id2))]

    def predict(self, pairs, bs=32):#batch size = 32.
      #function sends sentence pairs to the model and gets predictions.
        out = []
        with self.torch.no_grad():
            for i in range(0, len(pairs), bs):
                chunk = pairs[i:i + bs]
                #tokenizes the previous and current sentences as a pair.
                enc = self.tok([p for p, _ in chunk],
                 [c for _, c in chunk],
                               truncation=True,#Cuts text if it is too long.
                               max_length=CFG['max_tokens'], #	Maximum token length is 128.
                               padding=True,#	Makes the inputs in a batch the same length.
                               return_tensors='pt').to(self.dev)#Returns PyTorch tensors.
                out.append(self.torch.softmax(self.model(**enc).logits, -1).cpu().numpy())
                #Gets the model's raw classification scores.
                #Converts those scores into normalized confidence-like scores across the four labels.
        return np.vstack(out) if out else np.zeros((0, len(self.labels)))
#simply:This function gives two sentences to the fine-tuned IndicBERT model and gets its classification result.

def classify_sentences(sents, has_punct, lang, clf=None):
  #decides which classification method to use.
    """Returns (labels, confidences) per sentence. Uses the fine-tuned model if given, else the rules."""
    if clf is not None:
      #Fine-tuned model is available-->uses IndicBERT to classify the sentence pairs.
        pairs = [(sents[i - 1]['text'] if i else '', s['text']) for i, s in enumerate(sents)]
        P = clf.predict(pairs)
        idx = P.argmax(1)
        return [clf.labels[k] for k in idx], [float(P[i, k]) for i, k in enumerate(idx)]
        #Fine-tuned model is not available-->uses the rule-based weak-label function instead.
    labs = [weak_label(s['text'], lang, has_punct, s['n']) for s in sents]
    confs = [1.0 if l == 'neutral' else 0.8 if l == 'question' else 0.75 for l in labs]
    return labs, confs


# Moments -> segments [x - y, x + y]
#Finding moments and creating video clips
#project uses the classification results to identify important moments.
def find_moments(sents, labs, confs, cfg=CFG):
    out = []
    for i, s in enumerate(sents):
      #Question–Answer detection
        if labs[i] == 'question' and confs[i] >= cfg['q_thr']:
            ans, j, words = [], i + 1, 0
            while j < len(sents) and len(ans) < 3:
                nx = sents[j]
                if nx['start'] - s['end'] > (cfg['answer_window'] if not ans else 4):
                    break
                if labs[j] == 'question':
                    break
                ans.append(nx)
                words += nx['n']
                j += 1
            if words >= 4:
                conf = min(1.0, confs[i] + 0.1 * ans[0]['turn'])
                out.append({'type': 'Question-Answer', 'm_start': s['start'], 'm_end': ans[-1]['end'],
                            'conf': round(conf, 2),
                            'text': s['text'] + '  ->  ' + ' '.join(a['text'] for a in ans)})
                #If a sentence is classified as a question and its confidence reaches the required threshold, the function searches for:
                  # Whether the answer starts within 20 seconds.
                  # Whether another question appears.
                  # Whether there are enough answer words.
                  # Whether it can collect up to three following sentence groups.
                # system tries to combine the question and its answer into one detected moment.

        #Agreement and disagreement detection
        elif (labs[i] in ('agreement', 'disagreement') and confs[i] >= cfg['ad_thr'] and i > 0
              and s['n'] >= 2 and s['start'] - sents[i - 1]['end'] < 8):
            out.append({'type': labs[i].capitalize(), 'm_start': sents[i - 1]['start'], 'm_end': s['end'],
                        'conf': round(confs[i], 2), 'text': sents[i - 1]['text'] + '  ->  ' + s['text']})
      #If the sentence is classified as agreement or disagreement and its confidence reaches the required threshold, it checks the previous sentence.
      #It then combines the previous and current sentences into a conversation pair.
    return out


def make_segment(m_start, m_end, y, video_len, cfg=CFG):
  #function converts a detected moment into a video clip.
  #if detected moment (500 s) buffer time=25sec so (475:525 )sec
  #also ensures the clip stays within the minimum and maximum duration settings and does not go beyond the video boundaries.
    x = (m_start + m_end) / 2
    # find the middle part (avg) of the detected moment

    half = max(y, (m_end - m_start) / 2 + 2)
    # decides how many seconds should be included on each side of the middle point.
    # To make sure the clip includes enough time before and after the detected moment.

    half = min(max(half, cfg['min_len'] / 2), cfg['max_len'] / 2)
         #Apply minimum (30)and maximum limits(60)

    s, e = x - half, x + half
     # s means clip starting time.
    # e means clip ending time.
    if s < 0:
      #Check if the clip starts before the video
        e, s = e - s, 0.0
    if video_len and e > video_len:
        s, e = max(0.0, s - (e - video_len)), video_len
        #Check if the clip exceeds the video duration
    return s, e, x  # #Return the values(start time,endtime,middle point of the detected moment)


def _overlap(a, b):#Checking whether two clips overlap
    inter = max(0, min(a[1], b[1]) - max(a[0], b[0]))
    return inter / max(1e-9, min(a[1] - a[0], b[1] - b[0]))
     #If two clips do not overlap, the result is 0.
    #we dont need duplictaed clips


def select_segments(moments, y, video_len, cfg=CFG):
  # moments: all detected Question–Answer, Agreement and Disagreement moments.
  # y: buffer time.
  # video_len: original video duration.
    segs = [] #list stores all the video clips we calculate.
    for m in moments:
        s, e, x = make_segment(m['m_start'], m['m_end'], y, video_len, cfg)
        #the segs list will now contain the original information plus:
        #- x: middle point.
        #- seg_start: clip starting time.
        #- seg_end: clip ending time.
        #- **m = It copies all the existing information from m and adds the new information.
        segs.append({**m, 'x': x, 'seg_start': s, 'seg_end': e})
    segs.sort(key=lambda r: -r['conf'])
    #sorts clips from highest confidence to lowest confidence.
    #code wants to consider the most confident detections first.
    kept = [] #final selected clips
    for r in segs:
          # Remove overlapping clips of the same type

        if any(k['type'] == r['type'] and _overlap((k['seg_start'], k['seg_end']), (r['seg_start'], r['seg_end'])) > 0.5
               for k in kept):
            continue
        #It only removes overlapping clips when they have the same type.

        if sum(k['type'] == r['type'] for k in kept) >= cfg['top_k']:
            continue  #Limit the number of clips to 8 for each category

        kept.append(r)#Keep the selected clip
    return sorted(kept, key=lambda r: r['seg_start'])
    #Sort clips by starting time (chronological order)
    #Even though the code initially sorted by confidence, the final output is sorted by video time.

def fmt(t):
    t = int(t)
    return f'{t // 3600:02d}:{(t % 3600) // 60:02d}:{t % 60:02d}'
  #This function converts seconds into the familiar video timestamp format(HH:MM:SS)


CLIP_DIR = 'clips'


def ffmpeg_exe():
    """Path to ffmpeg: the one on PATH, otherwise the copy shipped with imageio-ffmpeg.
    FIXED: the bundled binary has an odd name (e.g. ffmpeg-win-x86_64-v7.1.exe) that yt-dlp does not recognise
    ("ffmpeg is not installed"). We copy it to ./ffmpeg_bin/ffmpeg(.exe) and add that folder to PATH."""
    exe = shutil.which('ffmpeg')
    if exe:
        return exe
    try:
        import imageio_ffmpeg
        src = imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return None
    bin_dir = os.path.abspath('ffmpeg_bin')
    dst = os.path.join(bin_dir, 'ffmpeg.exe' if os.name == 'nt' else 'ffmpeg')
    try:
        if not os.path.exists(dst):
            os.makedirs(bin_dir, exist_ok=True)
            shutil.copy2(src, dst)
            os.chmod(dst, 0o755)
        os.environ['PATH'] = bin_dir + os.pathsep + os.environ.get('PATH', '')
        return dst
    except Exception:
        return src


def clip_path(yt_id, start, end, out_dir=CLIP_DIR):
    return os.path.join(out_dir, f'{yt_id}_{int(start)}_{int(end)}.mp4')


def find_clip(yt_id, start, end, out_dir=CLIP_DIR):
    """Returns the path of an already-cut clip, or None."""
    f = clip_path(yt_id, start, end, out_dir)
    return f if os.path.exists(f) and os.path.getsize(f) > 10_000 else None


def clip_duration(path):
    """Length of a video file in seconds (read from ffmpeg's banner), or None."""
    exe = ffmpeg_exe()
    if not exe or not path:
        return None
    r = subprocess.run([exe, '-hide_banner', '-i', path], capture_output=True, text=True)
    m = re.search(r'Duration:\s*(\d+):(\d+):(\d+(?:\.\d+)?)', r.stderr)
    return int(m.group(1)) * 3600 + int(m.group(2)) * 60 + float(m.group(3)) if m else None


def make_clip(yt_id, start, end, out_dir=CLIP_DIR, max_height=480):
    """Cut seconds [start, end] of a YouTube video into a NEW .mp4 file and return its path.
    Only that section is downloaded. Raises RuntimeError with a readable message if it fails."""
    done = find_clip(yt_id, start, end, out_dir)
    if done:
        return done                                              # already cut earlier -> reuse
    try:
        import yt_dlp
        from yt_dlp.utils import download_range_func
    except ImportError:
        raise RuntimeError('yt-dlp is not installed. Run:  pip install -U yt-dlp')
    ff = ffmpeg_exe()
    if not ff:
        raise RuntimeError('ffmpeg not found. Run:  pip install imageio-ffmpeg   (or install ffmpeg and add it to PATH)')
    os.makedirs(out_dir, exist_ok=True)
    final = clip_path(yt_id, start, end, out_dir)
    stem = final[:-4]
    opts = {
        'format': (f'bv*[height<={max_height}][ext=mp4]+ba[ext=m4a]/b[height<={max_height}][ext=mp4]/'
                   f'bv*[height<={max_height}]+ba/b[height<={max_height}]/b'),
        'download_ranges': download_range_func(None, [(float(start), float(end))]),   # <- the actual cutting
        'force_keyframes_at_cuts': True,                         # frame-accurate start/end
        'merge_output_format': 'mp4',
        'outtmpl': stem + '.%(ext)s',
        'ffmpeg_location': ff,
        'overwrites': True, 'quiet': True, 'no_warnings': True, 'noprogress': True,
    }
    if os.environ.get('YT_COOKIES_BROWSER'):                     # e.g. set YT_COOKIES_BROWSER=chrome if YouTube asks to sign in
        opts['cookiesfrombrowser'] = (os.environ['YT_COOKIES_BROWSER'],)
    last_err = None
    for attempt in range(2):                                     # CHANGED: retry once (network hiccups, ffmpeg code 8)
        try:
            with yt_dlp.YoutubeDL(opts) as ydl:
                ydl.download([f'https://www.youtube.com/watch?v={yt_id}'])
            last_err = None
            break
        except Exception as e:
            last_err = e
    if last_err is not None:
        raise RuntimeError(f'yt-dlp could not download this section: {str(last_err)[:300]}')
    produced = [f for f in os.listdir(out_dir) if f.startswith(os.path.basename(stem) + '.') and not f.endswith('.part')]
    if not produced:
        raise RuntimeError('No clip file was produced.')
    src = os.path.join(out_dir, produced[0])
    if not src.lower().endswith('.mp4'):                         # webm/mkv -> mp4 so every browser can play it
        subprocess.run([ff, '-y', '-loglevel', 'error', '-i', src, '-c:v', 'libx264', '-c:a', 'aac', final], check=True)
        os.remove(src)
    return final


def _stream_info(yt_id, max_height=480):
    """FIXED: ONE YouTube lookup per video. YouTube often has NO single file with audio+video ("Requested format is
    not available"), so we ask for the best video (<= max_height) + best audio and return BOTH stream URLs."""
    import yt_dlp
    opts = {'format': f'bv*[height<={max_height}]+ba/b[height<={max_height}]/bv*+ba/b',
            'quiet': True, 'no_warnings': True, 'skip_download': True}
    if os.environ.get('YT_COOKIES_BROWSER'):
        opts['cookiesfrombrowser'] = (os.environ['YT_COOKIES_BROWSER'],)
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(f'https://www.youtube.com/watch?v={yt_id}', download=False)
    fmts = info.get('requested_formats') or [info]
    return [(f['url'], f.get('http_headers') or info.get('http_headers') or {}) for f in fmts]


def cut_from_stream(inputs, start, end, out_file):
    """FIXED: cut [start, end] seconds from one or two inputs (video stream + audio stream, URLs or local files)
    with ffmpeg and write a NEW mp4 (re-encoded, so the cut is exact)."""
    ff = ffmpeg_exe()
    if not ff:
        raise RuntimeError('ffmpeg not found. Run:  pip install imageio-ffmpeg')
    cmd = [ff, '-y', '-loglevel', 'error']
    for url, headers in inputs:
        if str(url).startswith('http'):
            if headers:
                cmd += ['-headers', ''.join(f'{k}: {v}\r\n' for k, v in headers.items())]
            cmd += ['-reconnect', '1', '-reconnect_streamed', '1', '-reconnect_delay_max', '5']
        cmd += ['-ss', str(start), '-i', url]
    cmd += ['-t', str(end - start)]
    if len(inputs) == 2:
        cmd += ['-map', '0:v:0', '-map', '1:a:0']                 # video from input 0, audio from input 1
    cmd += ['-c:v', 'libx264', '-preset', 'veryfast', '-crf', '23', '-c:a', 'aac', '-movflags', '+faststart', out_file]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        if os.path.exists(out_file):
            os.remove(out_file)
        raise RuntimeError(r.stderr[-300:])


def export_clips(df, out_dir=CLIP_DIR, progress=None, fast=True, max_per_type=None):
    """CHANGED (much faster): groups moments by video, does ONE YouTube lookup per video, then cuts every
    clip with ffmpeg. Falls back to the slow per-clip download (make_clip) if that fails.
    Resumable: clips that already exist are skipped.
    max_per_type: keep only the N most confident moments per (video, type); the others get clip_file = None."""
    d = df
    if max_per_type:
        d = df.sort_values('confidence', ascending=False).groupby(['yt_id', 'type']).head(max_per_type)
    os.makedirs(out_dir, exist_ok=True)
    paths, done, errs = {}, 0, []
    for yt_id, g in d.groupby('yt_id', sort=False):
        stream = None
        for idx, r in g.iterrows():
            s, e = int(r.seg_start), int(r.seg_end)
            f = find_clip(yt_id, s, e, out_dir)
            if f is None and fast and stream is not False:
                try:
                    if stream is None:
                        stream = _stream_info(yt_id)
                    cut_from_stream(stream, s, e, clip_path(yt_id, s, e, out_dir))
                    f = find_clip(yt_id, s, e, out_dir)
                except Exception as err:
                    print(f'  fast cut failed for {yt_id} ({str(err)[:120]}) -> slow fallback')
                    errs.append(str(err)[:200])
                    if stream is None:
                        stream = False                       # lookup itself failed: don't retry for this video
            if f is None:
                try:
                    f = make_clip(yt_id, s, e, out_dir)
                except Exception as err:
                    print(f'  clip {idx} failed: {str(err)[:150]}')
                    errs.append(str(err)[:200])
            paths[idx] = f
            done += 1
            if progress:
                progress(done, len(d))
        print(f'{yt_id}: {sum(1 for i in g.index if paths.get(i))}/{len(g)} clips ready')
    out = df.copy()
    out['clip_file'] = pd.Series([paths.get(i) for i in df.index], index=df.index, dtype=object)   # None when not cut
    out.attrs['errors'] = errs                                   # NEW: so the app can show WHY a clip failed
    return out


def zip_clips(paths, zip_path='clips.zip'):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in paths:
            if isinstance(p, str) and os.path.exists(p):
                z.write(p, os.path.basename(p))
    return zip_path



def analyse(yt_id, language='english', title='', video_id='', clf=None, cfg=CFG, snips=None):
  #connects the earlier functions.
    """Full pipeline for one video. Returns (DataFrame of segments, info dict)."""
    lang_used = None
    if snips is None:
        snips, lang_used = fetch_transcript(yt_id, language)
        # If transcript data has not already been provided, it retrieves captions from YouTube.
    if not snips:
        return pd.DataFrame(), {'error': 'no transcript'}
        # Check whether the transcript exists
        # If there is no transcript, it returns an empty table and an error message.
    sents, has_punct = build_sentences(snips)
    # Converts caption fragments into sentence-like groups.
    video_len = snips[-1]['start'] + snips[-1]['duration']
        #approximate total video duration using the final transcript fragment.
    labs, confs = classify_sentences(sents, has_punct, language, clf)
    #Uses IndicBERT if a fine-tuned classifier is supplied. Otherwise, it uses the rules.
    segs = select_segments(find_moments(sents, labs, confs, cfg),
                           cfg['buffer'],
                           video_len,
                           cfg)
    #Detects Question–Answer, Agreement and Disagreement moments and creates clip timings.
    rows = []
    for r in segs:
        in_seg = ' '.join(s['text'] for s in sents if s['start'] >= r['seg_start'] and s['end'] <= r['seg_end'])
        #Prepare the final results
        rows.append({'video_id': video_id,
                     'title': title,
                     'yt_id': yt_id,
                     'language': language,
                     'type': r['type'],
                     'moment_time': fmt(r['x']),
                     'seg_start': round(r['seg_start'], 1),
                     'seg_end': round(r['seg_end'], 1),
                     'duration_s': round(r['seg_end'] - r['seg_start'], 1),
                     'confidence': r['conf'],
                     'key_exchange': r['text'][:400],
                     'segment_transcript': in_seg})
    info = {'fragments': len(snips), 'sentences': len(sents), 'punctuated': has_punct, 'lang_used': lang_used,
            'video_len': video_len, 'label_counts': pd.Series(labs).value_counts().to_dict()}
    return pd.DataFrame(rows), info
#Returns the results as a Pandas DataFrame and additional information about the video.


# Training-data helper (weak labels -> sheet for human review)
#Preparing training data for fine-tuning
def build_training_candidates(stores, neutral_ratio=1.5, seed=0):
    """stores: list of dicts {video_id, language, sents, has_punct}. Returns a DataFrame to review."""
    #prepares a dataset that can review and correct before training IndicBERT.
    rows = []
    for st in stores:#Collect sentences
        S = st['sents']
        #Goes through the processed videos and their sentence-like groups.
        for i, s in enumerate(S):
            rows.append({'video_id': st['video_id'],
                         'language': st['language'],
                         'start': round(s['start'], 1),
                         'prev': S[i - 1]['text'] if i else '',
                         'cur': s['text'],
                         'weak_label': weak_label(s['text'], st['language'], st['has_punct'], s['n'])})
# Creates a table containing:
# - Video ID.
# - Language.
# - Timestamp.
# - Previous sentence.
# - Current sentence.
# - Automatically generated label.
    df = pd.DataFrame(rows)#df
    pos = df[df.weak_label != 'neutral']#contains question, agreement and disagreement
    neu = df[df.weak_label == 'neutral']#contains neutral
    neu = neu.sample(min(len(neu), int(len(pos) * neutral_ratio) + 50), random_state=seed)#reduces the number of neutral examples. (bias)
    out = pd.concat([pos, neu]).sort_values(['video_id', 'start']).reset_index(drop=True)#Combines the non-neutral and selected neutral examples.
    out['label'] = ''          # <- fill / correct this column by hand (leave blank to accept weak_label)
    return out


| Column | Meaning |
|---|---|
| `video_id` | Your video reference, such as V001 |
| `title` | Original YouTube video title |
| `yt_id` | YouTube video ID |
| `type` | Question–Answer, Agreement or Disagreement |
| `moment_time` | Middle time of the detected conversation |
| `seg_start` | Starting time of the selected clip |
| `seg_end` | Ending time of the selected clip |
| `duration_s` | Total clip duration in seconds |
| `confidence` | Confidence score from the detection stage |
| `key_exchange` | Short text showing the detected conversation |
| `segment_transcript` | Transcript of the selected clip |
| `clip_file` | Path of the cut `.mp4` clip for this moment (created by `pl.export_clips`) |

In [ ]:
#import libraries
import pandas as pd, json, os, time, numpy as np
from IPython.display import HTML, display
import pipeline as pl

EXCEL_PATH = 'video_metadata.xlsx'#file path
pl.CFG.update(#update
    buffer=25,          # y: segment = [x - y, x + y]
              min_len=30, max_len=60,   #               answer_window=20,   # answer must start within 20 s of the question
              q_thr=0.60, ad_thr=0.70, top_k=8)

videos = pd.read_excel(EXCEL_PATH) # read
videos['yt_id'] = videos['video_url'].apply(pl.extract_video_id)#function apply
skipped = videos[videos.yt_id.isna()]#null
if len(skipped): print('Skipping rows without a valid link:', list(skipped.video_id))#remove rows that don't have valid YouTube IDs.
videos = videos[videos.yt_id.notna()].reset_index(drop=True)#Resets the row numbering after removing invalid rows.
print(videos.groupby('language').size())#Counting videos by language
videos[['video_id','video_title','language','yt_id']]#display

language
English      5
Hindi        2
Malayalam    2
Tamil        2
Urdu         1
dtype: int64


,video_id,video_title,language,yt_id
0,V001,confidence isn't loud,English,rKpltaOMFdc
1,V002,Does money buy happiness,English,bdpyQm5l78o
2,V003,You cannot always be positive,English,-pYOlucclio
3,V004,We the women,English,63mazZL-Lh4
4,V005,The future mark zuckerberg is trying to build,English,oX7OduG1YmI
5,V006,Bollywood darkside nobody talk about,Hindi,EJXPeOKPG3Y
6,V007,Brain Rot is not Real,Hindi,u25wf-qmF0E
7,V008,Your mind is Easier to Fool than you think,Malayalam,cb_6ExZNwFc
8,V009,"Time travel,Multiverse & Interstellar",Malayalam,ViwK-gtSLgk
9,V010,Before you give up watch this!,Tamil,zEEP20TG43w


| Library | Purpose |
|---|---|
| `pandas` | Reads and processes Excel data. |
| `json` | Works with JSON data. |
| `os` | Handles files and folders. |
| `time` | Works with time-related operations. |
| `numpy` | Performs numerical operations. |
| `HTML, display` | Displays HTML content and results inside Google Colab. |
| `pipeline as pl` | Imports your `pipeline.py` functions. |

| Setting | Value | Meaning |
|---|---:|---|
| `buffer` | 25 | Adds approximately 25 seconds before and after the detected moment. |
| `min_len` | 30 | Minimum clip duration is 30 seconds. |
| `max_len` | 60 | Maximum clip duration is 60 seconds. |
| `answer_window` | 20 | Looks for the answer within 20 seconds after a question. |
| `q_thr` | 0.60 | Minimum confidence threshold for accepting a question prediction. |
| `ad_thr` | 0.70 | Minimum confidence threshold for accepting agreement/disagreement predictions. |
| `top_k` | 8 | Selects a maximum of 8 moments per category per video. |

In [ ]:
CACHE = 'transcripts.json'#stores the filename transcripts.json.
cache = json.load(open(CACHE, encoding='utf-8')) if os.path.exists(CACHE) else {}
#Checks whether the file transcripts.json exists and open,.read,load if not create empty dict
#encoding='utf-8' helps preserve different languages, including Malayalam, Hindi, Tamil and Urdu.
stores = []#empty list
# it will eventually contain:
# - Video ID
# - Language
# - YouTube ID
# - Video title
# - Original transcript fragments
# - Processed sentences
# - Punctuation information
# - Caption language used
for r in videos.itertuples():#Processing each video
#.itertuples() allows us to access each row's values using column names.
    if r.yt_id not in cache:# Checking whether the transcript is already saved
        snips, used = pl.fetch_transcript(r.yt_id, r.language)#Fetching the transcript
        cache[r.yt_id] = {'snips': snips, 'lang_used': used}#This stores the newly retrieved transcript inside the cache.
        json.dump(cache, open(CACHE, 'w', encoding='utf-8'), ensure_ascii=False) #saves the updated cache into transcripts.json.
    snips, used = cache[r.yt_id]['snips'], cache[r.yt_id]['lang_used']#Retrieving the transcript from the cache
    if not snips: print(f'{r.video_id} ({r.language}): NO CAPTIONS - see Whisper note at the end'); continue#Checking whether captions are available
    sents, hp = pl.build_sentences(snips)#function call
    stores.append(dict(video_id=r.video_id,
                       language=r.language,
                       yt_id=r.yt_id,
                       title=r.video_title,
                       snips=snips,#	Original YouTube transcript fragments.
                       sents=sents,#	Processed sentence-like segments.
                       has_punct=hp, #Punctuation information.
                       lang_used=used))#Actual caption language retrieved.
    flag = '' if (used is None or used.lower().startswith(pl.LANG_CODES[pl.lang_key(r.language)][0])) else f'  <-- WARNING: captions are "{used}"'
    #Checking whether the caption language matches
    print(f'{r.video_id} {r.language:10s} | {len(snips):5d} fragments -> {len(sents):4d} sentences | punctuated={hp} | caption lang={used}{flag}')
# Displaying transcript processing results is correct
for s in stores:
  #Checking whether Indic text is displayed correctly
    print(s['language'], '|', s['sents'][len(s['sents'])//2]['text'][:90])

V001 (English): NO CAPTIONS - see Whisper note at the end
V002 (English): NO CAPTIONS - see Whisper note at the end
V003 (English): NO CAPTIONS - see Whisper note at the end
V004 (English): NO CAPTIONS - see Whisper note at the end
V005 (English): NO CAPTIONS - see Whisper note at the end
V006 (Hindi): NO CAPTIONS - see Whisper note at the end
V007 (Hindi): NO CAPTIONS - see Whisper note at the end
V008 (Malayalam): NO CAPTIONS - see Whisper note at the end
V009 (Malayalam): NO CAPTIONS - see Whisper note at the end
V010 (Tamil): NO CAPTIONS - see Whisper note at the end
V011 (Tamil): NO CAPTIONS - see Whisper note at the end
V012 (Urdu): NO CAPTIONS - see Whisper note at the end


snips – Transcript fragments.

used – Actual caption language.

In [ ]:
#creates a dataset to review and correct before fine-tuning IndicBERT.
cand = pl.build_training_candidates(stores, neutral_ratio=1.5)#function call
#stores contains the processed sentences from your YouTube videos.
#contol nutal example selected
cand.to_csv('labels_to_review.csv', index=False, encoding='utf-8-sig')
print(cand.groupby(['language','weak_label']).size().unstack(fill_value=0))
#Display the number of examples in each category
try:
    from google.colab import files; files.download('labels_to_review.csv')
except Exception: pass
cand.head(10)

AttributeError: 'DataFrame' object has no attribute 'weak_label'

In [ ]:
#check reviewed CSV file exists
if os.path.exists('labels_to_review.csv'):
    lab = pd.read_csv('labels_to_review.csv', encoding='utf-8-sig').fillna({'prev': '', 'label': ''})#read
    lab['label'] = lab['label'].astype(str).str.strip().str.lower()#label clean
    lab.loc[lab.label.isin(['', 'nan']), 'label'] = lab['weak_label']
    #means any examples you haven't reviewed will keep their weak labels.
    print('Using human-reviewed labels')
else:
    lab = cand.copy(); lab['label'] = lab['weak_label']
    print('WARNING: labels_reviewed.csv not found -> training on WEAK labels only (demo run).')
assert set(lab.label) <= set(pl.LABELS), set(lab.label) - set(pl.LABELS)
lab['cur'] = lab['cur'].astype(str); lab['prev'] = lab['prev'].astype(str)
print(lab.groupby(['language', 'label']).size().unstack(fill_value=0))

In [ ]:
print("Total examples:", len(lab))

print("Labels different from weak labels:")
print((lab['label'] != lab['weak_label']).sum())

print("\nComparison:")
print(pd.crosstab(lab['weak_label'], lab['label']))

pd.read_csv() reads your reviewed CSV file.

encoding='utf-8-sig' helps preserve Malayalam, Hindi, Tamil and other language characters.

fillna() replaces missing values in prev and label with empty strings.

astype(str) converts values into strings.

str.strip() removes extra spaces.

str.lower() converts labels to lowercase.

##Training, Validation, and Testing.
Neighbouring sentences are near-duplicates in context, so a random split would leak. Within each video, the **first 70 %** of the timeline is train, the next **15 %** validation, the last **15 %** test.

In [ ]:
#Time-block splitting.
lab['pos'] = lab.groupby('video_id')['start'].rank(pct=True)
#Calculate the position of each sentence within its video
#pct=True gives the rank as a percentage between 0 and 1.
train_df = lab[lab.pos <= 0.70]; val_df = lab[(lab.pos > 0.70) & (lab.pos <= 0.85)]; test_df = lab[lab.pos > 0.85]
#train means the first 70% of the timeline from each video goes into the training dataset.
#validation selects the next 15% of the timeline.
#test means the last 15% of each video's timeline.
for n, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
  #Display the dataset counts
    print(n, len(d), d.label.value_counts().to_dict())

| Dataset | Percentage | Purpose |
|---|---:|---|
| Training | 70% | Teach the model |
| Validation | 15% | Check and improve the model during training |
| Testing | 15% | Evaluate the final model |

| Video timeline | Dataset |
|---|---|
| Beginning (0–70%) | Training |
| Middle (70–85%) | Validation |
| Ending (85–100%) | Testing |

##  Fine-tune IndicBERT v2 (`ai4bharat/IndicBERTv2-MLM-only`)


#IndicBERT version
IndicBERT v2 – MLM-only

#Model name:
ai4bharat/IndicBERTv2-MLM-only


This is a multilingual BERT model developed by AI4Bharat. It is pretrained using the Masked Language Modelling (MLM) objective.

`tokenizer` provided with IndicBERT v2 – MLM-only.

In [ ]:
#fine-tunes IndicBERT v2 using your labelled sentence pairs.

In [ ]:
import torch
import time
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          DataCollatorWithPadding)
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

BASE = 'ai4bharat/IndicBERTv2-MLM-only'      # pretrained model covers English + 22 Indian languages incl. Hindi, Tamil, Malayalam, Urdu
OUT  = 'models/indicbert_moments'#where we want to save that
lab2id = {l: i for i, l in enumerate(pl.LABELS)}; id2lab = {i: l for l, i in lab2id.items()}
#Convert category names into numbers
tok = AutoTokenizer.from_pretrained(BASE)#Load the tokenizer

class PairDS(Dataset):#Create a custom Dataset class
    def __init__(self, d):#d-train,test,val
        self.enc = tok(list(d.prev), list(d.cur), truncation=True, max_length=pl.CFG['max_tokens'])
# list(d.prev) – takes all previous sentences.
# list(d.cur) – takes all current sentences.
# tok() – sends both sentences to the tokenizer.
# truncation=True – cuts text if it exceeds the maximum allowed length.
# max_length=pl.CFG['max_tokens'] – limits the input length to the configured
        self.y = [lab2id[l] for l in d.label] #corrected label to tokens
    def __len__(self): return len(self.y)#Get the number of examples
    def __getitem__(self, i):#eg
        item = {k: v[i] for k, v in self.enc.items()}; item['labels'] = self.y[i]; return item
#Define the evaluation metrics
def compute_metrics(p):
    pred = p.predictions.argmax(-1)
    return {'macro_f1': f1_score(p.label_ids, pred, average='macro'), 'accuracy': accuracy_score(p.label_ids, pred)}
#Macro F1 calculates the F1 score for each of the four categories and takes their average.

#Prepare the three datasets
tr_ds, va_ds, te_ds = PairDS(train_df), PairDS(val_df), PairDS(test_df)
model = AutoModelForSequenceClassification.from_pretrained(BASE,
                                                           num_labels=len(pl.LABELS),
                                                           id2label=id2lab,
                                                           label2id=lab2id)
#Load IndicBERT for classification
# Calculate total steps for 4 epochs: (len(tr_ds) // batch_size) * epochs
batch_size = 16
total_steps = (len(tr_ds) // batch_size) * 4
warmup_steps = int(0.1 * total_steps)

args = TrainingArguments(output_dir='ckpt',
                         num_train_epochs=4,
                         per_device_train_batch_size=batch_size,
                         per_device_eval_batch_size=32,
                         learning_rate=3e-5, #0.00003
                         weight_decay=0.01,
                         warmup_steps=warmup_steps,
                         eval_strategy='epoch',
                         save_strategy='epoch',
                         load_best_model_at_end=True,
                         metric_for_best_model='macro_f1',
                         save_total_limit=1,
                         fp16=torch.cuda.is_available(),
                         report_to='none',
                         seed=42)
#Create the Trainer
trainer = Trainer(model=model,
                  args=args,
                  train_dataset=tr_ds,
                  eval_dataset=va_ds,
                  data_collator=DataCollatorWithPadding(tok),
                  compute_metrics=compute_metrics)
#Start the actual training
t0 = time.time(); trainer.train(); print(f'training time: {time.time()-t0:.0f}s')
trainer.save_model(OUT); tok.save_pretrained(OUT)
print('saved to', OUT)

model – IndicBERT.

args – training settings.

train_dataset – training examples.

eval_dataset – validation examples.

data_collator – handles padding so examples in a batch have matching lengths.

compute_metrics – calculates Macro F1 and accuracy.

| Library | Purpose |
|---|---|
| `torch` | Deep learning framework used to train the model |
| `time` | Measures how long training takes |
| `Dataset` | Organises sentence pairs and their labels |
| `AutoTokenizer` | Converts text into tokens that the model understands |
| `AutoModelForSequenceClassification` | Loads IndicBERT with a classification layer |
| `TrainingArguments` | Defines training settings |
| `Trainer` | Handles the training process |
| `DataCollatorWithPadding` | Makes input sequences in a batch the same length |
| `f1_score`, `accuracy_score` | Evaluate model performance |

| Parameter | Meaning |
|---|---|
| `output_dir='ckpt'` | Temporary training checkpoints are saved here |
| `num_train_epochs=4` | Train for four complete passes |
| `per_device_train_batch_size=16` | Process 16 training examples per batch |
| `per_device_eval_batch_size=32` | Process 32 validation examples per batch |
| `learning_rate=3e-5` | Controls how much model weights change during training |
| `weight_decay=0.01` | Helps reduce overfitting |
| `warmup_steps` | Gradually increases the learning rate at the beginning |
| `eval_strategy='epoch'` | Evaluate after each epoch |
| `save_strategy='epoch'` | Save a checkpoint after each epoch |
| `load_best_model_at_end=True` | Load the best checkpoint at the end |
| `metric_for_best_model='macro_f1'` | Select the best model using Macro F1 |
| `save_total_limit=1` | Keep only one checkpoint |
| `fp16` | Use half-precision training when CUDA is available |
| `report_to='none'` | Disable external experiment tracking |
| `seed=42` | Helps make random operations reproducible |

In [ ]:
# Evaluate the final best model on the test dataset
test_results = trainer.evaluate(eval_dataset=te_ds)

print("\nTest Results:")
print("Test Loss:", test_results['eval_loss'])
print("Test Accuracy:", test_results['eval_accuracy'])
print("Test Macro F1:", test_results['eval_macro_f1'])

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Get predictions for the test dataset
pred_output = trainer.predict(te_ds)

# Convert prediction scores into predicted class IDs
y_pred = np.argmax(pred_output.predictions, axis=1)

# Get actual labels
y_true = pred_output.label_ids

# Classification report
print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(pl.LABELS))),
    target_names=pl.LABELS,
    zero_division=0
))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(pl.LABELS)))
))

##Evaluate on the held-out test block (per class and per language)

In [ ]:
pred = trainer.predict(te_ds); yp = pred.predictions.argmax(-1); yt = pred.label_ids
print(classification_report(yt, yp, labels=list(range(4)), target_names=pl.LABELS, digits=3, zero_division=0))
print(pd.DataFrame(confusion_matrix(yt, yp, labels=list(range(4))), index=pl.LABELS, columns=pl.LABELS))
t = test_df.assign(ok=(yt == yp))
print(t.groupby('language').apply(lambda g: pd.Series({'n': len(g), 'accuracy': g.ok.mean()})))
# Baseline = the rule-based weak label itself. Only meaningful if labels were human-reviewed.
print('rule baseline macro-F1 :', round(f1_score(test_df.label, test_df.weak_label, average='macro', labels=pl.LABELS, zero_division=0), 3))
print('fine-tuned macro-F1    :', round(f1_score(yt, yp, average='macro', labels=list(range(4)), zero_division=0), 3))

##  Extract moments from every video with the fine-tuned model

In [ ]:
clf = pl.MomentClassifier(OUT)
frames = []
for r in videos.itertuples():
    snips = cache[r.yt_id]['snips']
    if not snips: continue
    df, info = pl.analyse(r.yt_id, r.language, r.video_title, r.video_id, clf=clf, snips=snips)
    print(r.video_id, r.language, info['label_counts'])
    frames.append(df)
results = pd.concat([f for f in frames if len(f)], ignore_index=True)
results.to_csv('extracted_moments.csv', index=False, encoding='utf-8-sig')
print(results.groupby(['language', 'type']).size().unstack(fill_value=0))
results[['video_id','language','type','moment_time','seg_start','seg_end','confidence','key_exchange']]

## 8b. Cut every moment into its own video clip
`pl.export_clips` cuts each moment (`seg_start` -> `seg_end`, 30-60 s) into a **separate `.mp4` file** in `clips/`. Only the cut clips are played - the full video is never shown. The Streamlit app uses the same function.

In [ ]:
# OPTIONAL: removes the 'No supported JavaScript runtime' warning (it can also make some failed videos work)
!pip -q install -U "yt-dlp[default]"
!curl -fsSL https://deno.land/install.sh | sh > /dev/null 2>&1
import os; os.environ['PATH'] += ':/root/.deno/bin'

In [ ]:
import importlib; importlib.reload(pl)      # use the new pipeline.py (run the %%writefile pipeline.py cell first)
# FAST: one YouTube lookup per video, then ffmpeg cuts each clip. Resumable - clips already in clips/ are skipped.
# max_per_type=3 keeps the 3 most confident moments per (video, type); use None to cut all of them.
results = pl.export_clips(results, max_per_type=3)
results.to_csv('extracted_moments.csv', index=False, encoding='utf-8-sig')
print(results.clip_file.notna().sum(), 'of', len(results), 'clips created')
results[results.clip_file.notna()][['video_id','language','type','seg_start','seg_end','clip_file']]

In [ ]:
# Play a cut clip inside Colab (the file is a real 30-60 s video, so it cannot run past its end)
from IPython.display import Video
def play_clip(i): return Video(results.iloc[i].clip_file, embed=True, width=640)
play_clip(0)

## 8c. Try any YouTube link: get one cut clip per moment
Give a link and a language; you get the Question-Answer, Agreement and Disagreement clips as separate videos (the same thing the Streamlit app does).

In [ ]:
from IPython.display import Video, display
def clips_from_link(url, language='English', max_per_type=2):
    yt_id = pl.extract_video_id(url)
    res, info = pl.analyse(yt_id, language, title=url, video_id=yt_id, clf=clf)
    cut = pl.export_clips(res, max_per_type=max_per_type)
    for t in ['Question-Answer', 'Agreement', 'Disagreement']:
        for r in cut[(cut.type == t) & cut.clip_file.notna()].sort_values('confidence', ascending=False).itertuples():
            print(f'{t} | conf {r.confidence:.2f} | from {r.moment_time} of the video\n{r.key_exchange}')
            display(Video(r.clip_file, embed=True, width=520))
    return cut
# cut = clips_from_link('https://www.youtube.com/watch?v=XXXXXXXXXXX', 'Hindi')

In [ ]:
!zip -qr indicbert_moments.zip models/indicbert_moments
# from google.colab import drive; drive.mount('/content/drive'); !cp indicbert_moments.zip /content/drive/MyDrive/
# Or: from huggingface_hub import notebook_login; notebook_login(); clf.model.push_to_hub('YOUR_NAME/indicbert-moments'); clf.tok.push_to_hub('YOUR_NAME/indicbert-moments')